# Trabalho Final Pratico - Python & Pandas

## Tema 1: Desempenho Academico de Estudantes

Base principal: `student-por.csv` - desempenho em Portugues, com **649 registros** e **33 colunas originais**.  
Base complementar: `student-mat.csv` - desempenho em Matematica, usada apenas para comparacao opcional entre disciplinas.

**Objetivo:** analisar como habitos de estudo, reprovacoes anteriores, apoio familiar, escola, sexo, consumo de alcool, faltas e escolaridade dos pais se relacionam com a nota final `G3`.


## Como executar este notebook no PyCharm

Este trabalho foi organizado para ser executado como notebook Jupyter (`.ipynb`) dentro do PyCharm, mantendo celulas Markdown e celulas de codigo.

Antes de executar:

1. Abra no PyCharm a pasta do projeto `Projeto_Final_Novas_Tecnologias`.
2. Configure o interpretador Python do projeto.
3. Instale as bibliotecas com `pip install -r requirements.txt`.
4. Abra o arquivo `projeto_final.ipynb`.
5. Execute as celulas em ordem, do inicio ao fim.

Os arquivos `student-por.csv` e `student-mat.csv` precisam estar na mesma pasta do notebook. Se aparecer erro de arquivo nao encontrado, confira o *working directory* do PyCharm.


## 0. Configuracao do ambiente

Nesta etapa importamos as bibliotecas principais. O trabalho usa Pandas e NumPy para manipulacao dos dados, Matplotlib e Seaborn para visualizacoes e `Path` para localizar os arquivos na mesma pasta do notebook.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

pd.set_option('display.max_columns', 80)
pd.set_option('display.max_rows', 120)
pd.set_option('display.float_format', '{:.2f}'.format)

sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 11


## 1. Carregamento dos dados

Os arquivos CSV da base Student Performance usam ponto e virgula como separador. Por isso, a leitura correta precisa informar `sep=';'`.


In [ ]:
BASE_DIR = Path.cwd()
ARQ_PORTUGUES = BASE_DIR / 'student-por.csv'
ARQ_MATEMATICA = BASE_DIR / 'student-mat.csv'

if not ARQ_PORTUGUES.exists() or not ARQ_MATEMATICA.exists():
    raise FileNotFoundError(
        'Arquivos CSV nao encontrados. No PyCharm, abra a pasta do projeto como raiz '
        'ou ajuste o working directory para a pasta onde estao student-por.csv e student-mat.csv.'
    )

# Base principal do trabalho: Portugues, pois possui 649 registros como descrito no Tema 1.
df_raw = pd.read_csv(ARQ_PORTUGUES, sep=';', encoding='utf-8')

# Base complementar: Matematica, usada apenas para uma comparacao final opcional.
df_mat = pd.read_csv(ARQ_MATEMATICA, sep=';', encoding='utf-8')

print('Pasta de execucao:', BASE_DIR)
print('Base principal - Portugues:', df_raw.shape)
print('Base complementar - Matematica:', df_mat.shape)
display(df_raw.head())


A base de Portugues atende aos requisitos minimos do trabalho: possui **649 registros**, **33 colunas**, mais de 4 variaveis numericas e mais de 3 variaveis categoricas.


## 2. Compreensao inicial da base

Aqui verificamos dimensoes, tipos de dados, estatisticas descritivas e cardinalidade das variaveis categoricas. Essa etapa ajuda a entender a qualidade e a estrutura dos dados antes das analises.


In [ ]:
print('Dimensoes:', df_raw.shape)
print()
print('Informacoes gerais:')
df_raw.info()

print()
print('Estatisticas das variaveis numericas:')
display(df_raw.describe().T)

print()
print('Cardinalidade das variaveis categoricas:')
cardinalidade = df_raw.select_dtypes(include=['object', 'string']).nunique().sort_values(ascending=False)
display(cardinalidade.to_frame('quantidade_de_categorias'))


A nota final `G3` varia de 0 a 20. Na base original, a media de `G3` e **11.91**, a mediana e **12.00** e o desvio padrao e **3.23**, indicando uma distribuicao concentrada perto da faixa de aprovacao.


## 3. Dicionario de dados

A tabela abaixo resume as variaveis mais importantes para responder as perguntas do Tema 1.


In [ ]:
dicionario = pd.DataFrame([
    ['school', 'Escola do estudante: GP ou MS', 'Categorica', 'Comparar desempenho por escola'],
    ['sex', 'Sexo do estudante: F ou M', 'Categorica', 'Comparar desempenho por sexo'],
    ['age', 'Idade do estudante', 'Numerica discreta', 'Criar faixas etarias'],
    ['Medu', 'Escolaridade da mae, de 0 a 4', 'Numerica ordinal', 'Contexto familiar'],
    ['Fedu', 'Escolaridade do pai, de 0 a 4', 'Numerica ordinal', 'Contexto familiar'],
    ['studytime', 'Tempo semanal de estudo, de 1 a 4', 'Numerica ordinal', 'Pergunta 1'],
    ['failures', 'Numero de reprovacoes anteriores', 'Numerica discreta', 'Pergunta 2 e risco academico'],
    ['famsup', 'Apoio educacional da familia', 'Categorica', 'Pergunta 3'],
    ['Dalc', 'Consumo de alcool em dias uteis, de 1 a 5', 'Numerica ordinal', 'Pergunta 5'],
    ['Walc', 'Consumo de alcool no fim de semana, de 1 a 5', 'Numerica ordinal', 'Pergunta 5'],
    ['absences', 'Numero de faltas', 'Numerica discreta', 'Risco academico e outliers'],
    ['G1', 'Nota do primeiro periodo', 'Numerica', 'Historico de desempenho'],
    ['G2', 'Nota do segundo periodo', 'Numerica', 'Historico de desempenho'],
    ['G3', 'Nota final', 'Numerica', 'Variavel-alvo da analise'],
], columns=['Coluna', 'Descricao', 'Tipo', 'Papel na analise'])

display(dicionario)


## 4. Limpeza e tratamento

A limpeza verifica valores ausentes, duplicados e textos inconsistentes. Mesmo quando nao ha nulos, o notebook demonstra a estrategia de tratamento com `fillna()` e `dropna()` para cumprir o processo tecnico esperado.


In [ ]:
# Copia de trabalho para preservar a base original.
df_limpo = df_raw.copy()

# 1) Valores ausentes
nulos_por_coluna = df_limpo.isna().sum().sort_values(ascending=False)
print('Total de valores ausentes:', int(nulos_por_coluna.sum()))
display(nulos_por_coluna[nulos_por_coluna > 0].to_frame('nulos'))

# 2) Duplicados
duplicados_antes = df_limpo.duplicated().sum()
print('Duplicados antes da limpeza:', int(duplicados_antes))
df_limpo = df_limpo.drop_duplicates()

# 3) Linhas completamente vazias, caso existam
df_limpo = df_limpo.dropna(how='all')

# 4) Padronizacao de texto com strip, lower e upper.
colunas_texto = df_limpo.select_dtypes(include=['object', 'string']).columns
df_limpo[colunas_texto] = df_limpo[colunas_texto].apply(lambda coluna: coluna.str.strip().str.lower())
df_limpo['school'] = df_limpo['school'].str.upper()
df_limpo['sex'] = df_limpo['sex'].str.upper()

# 5) Preenchimento de nulos caso aparecam em outra versao da base
colunas_numericas = df_limpo.select_dtypes(include='number').columns
colunas_texto = df_limpo.select_dtypes(include=['object', 'string']).columns
df_limpo[colunas_numericas] = df_limpo[colunas_numericas].fillna(df_limpo[colunas_numericas].median())
df_limpo[colunas_texto] = df_limpo[colunas_texto].fillna('nao_informado')

# 6) Demonstracao de pd.to_datetime: a base nao possui data real, entao criamos metadado de processamento.
df_limpo['data_processamento'] = pd.to_datetime('2026-05-26')
df_limpo['ano_processamento'] = df_limpo['data_processamento'].dt.year

print('Dimensoes apos limpeza:', df_limpo.shape)
display(df_limpo.head())


A base nao apresentou valores ausentes (**0 nulos**) nem registros duplicados (**0 duplicados**). A coluna `data_processamento` foi criada apenas para demonstrar `pd.to_datetime()`, pois a base Student Performance nao possui uma variavel temporal real.


## 5. Transformacao e criacao de indicadores derivados

Nesta etapa criamos colunas uteis para analise: aprovacao, reprovacao anterior, consumo medio de alcool, media parcial, faixas de faltas, faixa etaria, nivel educacional dos pais e indice de risco academico.


In [ ]:
df = df_limpo.copy()

# Aprovacao: como as notas vao de 0 a 20, usamos 10 como ponto de corte.
df['aprovado'] = np.where(df['G3'] >= 10, 'Aprovado', 'Reprovado')

# Reprovacao anterior.
df['teve_reprovacao'] = np.where(df['failures'] > 0, 'Sim', 'Nao')

# Consumo medio de alcool combinando dias uteis e fim de semana.
df['alcool_total'] = df[['Dalc', 'Walc']].mean(axis=1)

# Media parcial antes da nota final. Evita usar G3 dentro do indice de risco que depois sera comparado com G3.
df['media_parcial'] = df[['G1', 'G2']].mean(axis=1)

# Faixas numericas com pd.cut.
df['faixa_faltas'] = pd.cut(
    df['absences'],
    bins=[-1, 0, 5, 15, df['absences'].max()],
    labels=['Sem faltas', '1 a 5', '6 a 15', 'Mais de 15']
)

df['faixa_etaria'] = pd.cut(
    df['age'],
    bins=[14, 16, 18, 22],
    labels=['15-16', '17-18', '19+']
)

# Educacao media dos pais.
df['educ_pais_media'] = df[['Medu', 'Fedu']].mean(axis=1)
df['nivel_educ_pais'] = pd.cut(
    df['educ_pais_media'],
    bins=[-0.1, 1.49, 2.49, 4],
    labels=['Baixa', 'Media', 'Alta']
)

# Quartis com pd.qcut, usando a media parcial como variavel academica anterior ao resultado final.
df['quartil_media_parcial'] = pd.qcut(df['media_parcial'], q=4, duplicates='drop')

# Classificacao de perfil de estudo com np.select.
df['perfil_estudo'] = np.select(
    [df['studytime'] <= 1, df['studytime'].eq(2), df['studytime'] >= 3],
    ['Baixo', 'Intermediario', 'Alto'],
    default='Nao classificado'
)

# Indice de risco academico com apply + lambda.
df['score_risco'] = df.apply(
    lambda linha: int(linha['media_parcial'] < 10) * 3
    + int(linha['failures'] > 0) * 2
    + int(linha['absences'] > 10)
    + int(linha['studytime'] <= 1)
    + int(linha['alcool_total'] >= 3),
    axis=1
)

df['risco_academico'] = pd.cut(
    df['score_risco'],
    bins=[-1, 1, 3, 8],
    labels=['Baixo', 'Medio', 'Alto']
)

display(df[['G1', 'G2', 'G3', 'media_parcial', 'aprovado', 'failures', 'teve_reprovacao', 'absences', 'faixa_faltas', 'alcool_total', 'score_risco', 'risco_academico']].head(10))


O `score_risco` combina sinais disponiveis antes ou durante o processo escolar: notas parciais, reprovacoes anteriores, muitas faltas, baixo tempo de estudo e consumo medio de alcool mais alto. Esse indice nao prova causalidade, mas ajuda a organizar perfis de maior atencao.


## 6. Outliers com metodo IQR

O metodo IQR foi aplicado a duas variaveis numericas: `absences` e `G3`. Em vez de remover automaticamente os casos extremos, primeiro quantificamos e interpretamos o que eles representam.


In [ ]:
def resumo_iqr(dataframe, coluna):
    q1 = dataframe[coluna].quantile(0.25)
    q3 = dataframe[coluna].quantile(0.75)
    iqr = q3 - q1
    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr
    mascara_outlier = (dataframe[coluna] < limite_inferior) | (dataframe[coluna] > limite_superior)
    return {
        'coluna': coluna,
        'q1': q1,
        'q3': q3,
        'iqr': iqr,
        'limite_inferior': limite_inferior,
        'limite_superior': limite_superior,
        'qtd_outliers': int(mascara_outlier.sum()),
        'pct_outliers': mascara_outlier.mean() * 100
    }

outliers = pd.DataFrame([
    resumo_iqr(df, 'absences'),
    resumo_iqr(df, 'G3')
])

display(outliers)


Pelo IQR, `absences` tem **21 outliers** acima de 15 faltas. Em `G3`, aparecem **16 outliers**, concentrados em notas muito baixas. Como esses valores podem representar situacoes academicas reais, a analise principal mantem os registros e apenas destaca sua influencia.


## 7. Pergunta 1 - Relacao entre tempo de estudo e nota final

A primeira pergunta verifica se estudantes com mais tempo semanal de estudo (`studytime`) apresentam maior nota final (`G3`).


In [ ]:
analise_estudo = df.groupby('studytime').agg(
    quantidade=('G3', 'size'),
    media_G3=('G3', 'mean'),
    mediana_G3=('G3', 'median'),
    aprovacao_pct=('aprovado', lambda s: (s == 'Aprovado').mean() * 100)
).round(2)

display(analise_estudo)


In [ ]:
rotulos_estudo = {1: '<2h', 2: '2-5h', 3: '5-10h', 4: '>10h'}
media_estudo = df.groupby('studytime')['G3'].mean().rename(index=rotulos_estudo)

ax = media_estudo.plot(kind='bar', color='#4C78A8', edgecolor='black')
ax.set_title('Media da nota final por tempo semanal de estudo')
ax.set_xlabel('Tempo semanal de estudo')
ax.set_ylabel('Media de G3')
ax.bar_label(ax.containers[0], fmt='%.2f')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


**Interpretacao:** existe uma tendencia positiva: estudantes no nivel 1 de estudo tiveram media **10.84**, enquanto o nivel 3 chegou a **13.23**. O nivel 4 ficou muito proximo (**13.06**), sugerindo que estudar mais esta associado a melhor desempenho, embora o ganho nao cresca indefinidamente.


## 8. Pergunta 2 - Reprovacoes anteriores e desempenho

Aqui comparamos estudantes com e sem reprovacoes anteriores (`failures > 0`).


In [ ]:
analise_reprovacao = df.groupby('teve_reprovacao').agg(
    quantidade=('G3', 'size'),
    media_G3=('G3', 'mean'),
    mediana_G3=('G3', 'median'),
    aprovacao_pct=('aprovado', lambda s: (s == 'Aprovado').mean() * 100),
    faltas_medias=('absences', 'mean')
).round(2)

display(analise_reprovacao)


**Interpretacao:** estudantes sem reprovacao anterior tiveram media **12.51** em `G3`, contra **8.59** entre os que ja reprovaram. Essa e uma das diferencas mais fortes observadas, o que torna `failures` uma variavel importante para identificar risco academico.


## 9. Pergunta 3 - Apoio familiar e aprovacao

A variavel `famsup` indica se o estudante recebe apoio educacional da familia.


In [ ]:
analise_famsup = df.groupby('famsup').agg(
    quantidade=('G3', 'size'),
    media_G3=('G3', 'mean'),
    mediana_G3=('G3', 'median'),
    aprovacao_pct=('aprovado', lambda s: (s == 'Aprovado').mean() * 100)
).round(2)

tab_famsup = pd.crosstab(df['famsup'], df['aprovado'], normalize='index').mul(100).round(2)

display(analise_famsup)
display(tab_famsup)


**Interpretacao:** estudantes com apoio familiar tiveram media **12.06** e aprovacao de **85.68%**. Sem apoio familiar, a media foi **11.67** e a aprovacao **82.87%**. A diferenca existe, mas e moderada.


## 10. Pergunta 4 - Diferencas por escola e sexo

Esta analise usa `groupby` multinivel com duas dimensoes (`school` e `sex`) e multiplas metricas.


In [ ]:
analise_escola_sexo = df.groupby(['school', 'sex']).agg(
    quantidade=('G3', 'size'),
    media_G3=('G3', 'mean'),
    mediana_G3=('G3', 'median'),
    desvio_G3=('G3', 'std'),
    aprovacao_pct=('aprovado', lambda s: (s == 'Aprovado').mean() * 100),
    faltas_medias=('absences', 'mean')
).round(2)

display(analise_escola_sexo)


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
df.boxplot(column='G3', by='school', ax=ax, grid=False)
ax.set_title('Distribuicao da nota final por escola')
ax.set_xlabel('Escola')
ax.set_ylabel('Nota final G3')
plt.suptitle('')
plt.tight_layout()
plt.show()


**Interpretacao:** a combinacao com melhor media foi `GP/F`, com **13.00**. A menor media apareceu em `MS/M`, com **9.95**. Isso sugere diferencas relevantes entre escolas e grupos, mas a analise nao permite afirmar que escola ou sexo sejam causas isoladas do desempenho.


## 11. Pergunta 5 - Consumo de alcool e nota final

As variaveis `Dalc` e `Walc` medem consumo de alcool em dias uteis e fins de semana. A analise calcula correlacoes com `G3` e tambem observa medias por nivel de consumo.


In [ ]:
correlacao_alcool = df[['Dalc', 'Walc', 'alcool_total', 'G3']].corr()['G3'].drop('G3').sort_values()
analise_alcool = df.groupby('Dalc').agg(
    quantidade=('G3', 'size'),
    media_G3=('G3', 'mean'),
    aprovacao_pct=('aprovado', lambda s: (s == 'Aprovado').mean() * 100)
).round(2)

display(correlacao_alcool.to_frame('correlacao_com_G3'))
display(analise_alcool)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(data=df, x='absences', y='G3', hue='risco_academico', alpha=0.7, ax=ax)
ax.set_title('Faltas e nota final por nivel de risco academico')
ax.set_xlabel('Numero de faltas')
ax.set_ylabel('Nota final G3')
ax.legend(title='Risco academico')
plt.tight_layout()
plt.show()


**Interpretacao:** as correlacoes entre alcool e nota final sao negativas, mas fracas: `Dalc` = **-0.205**, `Walc` = **-0.177** e `alcool_total` = **-0.209**. Isso indica associacao negativa, nao causalidade; outros fatores como estudo, historico de notas e reprovacoes parecem mais fortes.


## 12. Pergunta 6 - Indice de risco academico

O indice de risco academico combina media parcial, reprovacoes, faltas, pouco estudo e consumo medio de alcool. O objetivo e criar uma visao resumida de estudantes que podem exigir mais acompanhamento.


In [ ]:
analise_risco = df.groupby('risco_academico', observed=True).agg(
    quantidade=('G3', 'size'),
    media_G3=('G3', 'mean'),
    mediana_G3=('G3', 'median'),
    aprovacao_pct=('aprovado', lambda s: (s == 'Aprovado').mean() * 100),
    faltas_medias=('absences', 'mean')
).round(2)

display(analise_risco)


**Interpretacao:** o grupo de risco baixo teve media **13.47** e aprovacao de **99.75%**. Ja o grupo de risco alto teve media **8.39** e aprovacao de **41.73%**. Como o indice usa `G1` e `G2`, a comparacao com `G3` fica mais coerente do que usar diretamente a nota final dentro do proprio indice.


## 13. Pergunta 7 - Correlacoes mais fortes com a nota final

A matriz de correlacao identifica quais variaveis numericas estao mais associadas a nota final `G3`.


In [ ]:
colunas_corr = [
    'age', 'Medu', 'Fedu', 'traveltime', 'studytime', 'failures',
    'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health',
    'absences', 'G1', 'G2', 'G3', 'alcool_total',
    'media_parcial', 'educ_pais_media', 'score_risco'
]

corr = df[colunas_corr].corr()
correlacoes_g3 = corr['G3'].drop('G3').sort_values(key=lambda s: s.abs(), ascending=False)

display(correlacoes_g3.head(10).to_frame('correlacao_com_G3'))


In [ ]:
fig, ax = plt.subplots(figsize=(13, 10))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, linewidths=0.5, ax=ax)
ax.set_title('Mapa de calor das correlacoes entre variaveis numericas')
plt.tight_layout()
plt.show()


**Interpretacao:** as maiores correlacoes com `G3` foram `G2` (**0.919**), `media_parcial` (**0.905**) e `G1` (**0.826**), o que e esperado porque representam notas anteriores. Entre fatores de contexto, `failures` teve correlacao negativa de **-0.393**. Correlacao nao implica causalidade, mas ajuda a priorizar variaveis para investigacao.


## 14. Pergunta 8 - Tabela pivo por escola e escolaridade dos pais

A tabela dinamica compara a media de `G3` por escola e nivel medio de escolaridade dos pais.


In [ ]:
pivot_escola_educacao = pd.pivot_table(
    df,
    values='G3',
    index='school',
    columns='nivel_educ_pais',
    aggfunc='mean',
    observed=True
).round(2)

display(pivot_escola_educacao)


**Interpretacao:** na escola `GP`, estudantes com nivel educacional familiar alto tiveram media **12.99**. Na escola `MS`, o grupo com baixa escolaridade familiar teve media **9.93**. A tabela sugere que escola e contexto familiar podem se combinar na explicacao do desempenho.


## 15. Visualizacoes obrigatorias restantes

As proximas visualizacoes completam os 7 tipos exigidos: histograma, linha e barras empilhadas. Barras, dispersao, box plot e heatmap ja foram produzidos nas secoes anteriores.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
df['G3'].plot(kind='hist', bins=20, edgecolor='white', color='#59A14F', ax=ax)
ax.axvline(df['G3'].mean(), color='red', linestyle='--', label=f"Media: {df['G3'].mean():.2f}")
ax.axvline(df['G3'].median(), color='black', linestyle=':', label=f"Mediana: {df['G3'].median():.2f}")
ax.set_title('Distribuicao da nota final G3')
ax.set_xlabel('Nota final G3')
ax.set_ylabel('Frequencia')
ax.legend()
plt.tight_layout()
plt.show()


**Interpretacao:** a distribuicao de `G3` esta concentrada perto de 10 a 14, com media **11.91** e mediana **12.00**. A assimetria negativa (**-0.91**) indica presenca de algumas notas muito baixas puxando a media para baixo.


In [ ]:
notas_periodos = df[['G1', 'G2', 'G3']].mean().rename(index={
    'G1': '1o periodo',
    'G2': '2o periodo',
    'G3': 'Nota final'
})

fig, ax = plt.subplots(figsize=(8, 5))
notas_periodos.plot(kind='line', marker='o', color='#E15759', ax=ax)
ax.set_title('Evolucao media das notas ao longo dos periodos')
ax.set_xlabel('Periodo')
ax.set_ylabel('Media da nota')
ax.set_ylim(0, 20)
plt.tight_layout()
plt.show()


**Interpretacao:** o grafico de linha nao representa uma serie temporal com datas, e sim a sequencia academica `G1 -> G2 -> G3`. Ele mostra se a media da turma melhora, piora ou se mantem estavel ao longo dos tres periodos avaliados.


In [ ]:
tab_empilhada = pd.crosstab(df['school'], df['aprovado'], normalize='index').mul(100)

ax = tab_empilhada.plot(kind='bar', stacked=True, color=['#E15759', '#59A14F'], edgecolor='black')
ax.set_title('Proporcao de aprovacao e reprovacao por escola')
ax.set_xlabel('Escola')
ax.set_ylabel('Percentual de estudantes')
ax.legend(title='Situacao', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


**Interpretacao:** as barras empilhadas facilitam comparar a composicao de aprovados e reprovados dentro de cada escola. Esse tipo de grafico e mais adequado que barras simples quando o foco e proporcao entre categorias.


## 16. Interpretacao estatistica: media, mediana, desvio padrao e assimetria

A rubrica pede interpretacao estatistica de pelo menos duas variaveis. Aqui usamos `G3` e `absences`.


In [ ]:
estatisticas_chave = df[['G3', 'absences']].agg(['mean', 'median', 'std', 'skew']).T.round(2)
display(estatisticas_chave)


**Interpretacao:** em `G3`, media (**11.91**) e mediana (**12.00**) sao proximas, mas a assimetria de **-0.91** mostra cauda para notas baixas. Em `absences`, a media (**3.66**) e maior que a mediana (**2.00**) e a assimetria e **2.02**, indicando que poucos estudantes com muitas faltas elevam a media.


## 17. Indicadores sinteticos

A tabela final resume 7 indicadores pedidos no trabalho.


In [ ]:
indicadores = pd.DataFrame({
    'Indicador': [
        'Media geral de G3',
        'Mediana geral de G3',
        'Desvio padrao de G3',
        '% de aprovacao',
        '% de reprovacao',
        '% com faltas altas (>10)',
        'Melhor perfil de estudo'
    ],
    'Valor': [
        f"{df['G3'].mean():.2f}",
        f"{df['G3'].median():.2f}",
        f"{df['G3'].std():.2f}",
        f"{(df['aprovado'] == 'Aprovado').mean() * 100:.2f}%",
        f"{(df['aprovado'] == 'Reprovado').mean() * 100:.2f}%",
        f"{(df['absences'] > 10).mean() * 100:.2f}%",
        df.groupby('perfil_estudo')['G3'].mean().idxmax()
    ]
})

display(indicadores.set_index('Indicador'))


Os indicadores mostram um desempenho geral positivo: media **11.91**, aprovacao de **84.59%** e reprovacao de **15.41%**. Apenas **7.55%** dos estudantes tiveram mais de 10 faltas, e o melhor perfil de estudo foi **Alto**, com media aproximada de **13.18**.


## 18. Comparacao opcional com Matematica

A base de Matematica tem menos de 500 registros, entao nao foi escolhida como base principal. Mesmo assim, ela pode ser usada como comparacao complementar entre disciplinas.


In [ ]:
comparacao_disciplinas = pd.DataFrame({
    'disciplina': ['Portugues', 'Matematica'],
    'registros': [len(df), len(df_mat)],
    'media_G3': [df['G3'].mean(), df_mat['G3'].mean()],
    'aprovacao_pct': [(df['G3'] >= 10).mean() * 100, (df_mat['G3'] >= 10).mean() * 100]
}).round(2)

display(comparacao_disciplinas)


**Interpretacao:** Portugues teve media **11.91** e aprovacao de **84.59%**. Matematica teve media **10.42** e aprovacao de **67.09%**. Essa comparacao e complementar, porque as bases nao devem ser simplesmente somadas sem cuidado: ha estudantes que aparecem nas duas disciplinas.


## 19. Conclusoes e limitacoes

A analise indica que o desempenho final esta fortemente associado ao historico de notas: `G2` teve correlacao de **0.919** com `G3`, `media_parcial` teve correlacao de **0.905** e `G1` teve correlacao de **0.826**. Isso mostra que o desempenho anterior e o principal sinal da nota final.

As reprovacoes anteriores tambem aparecem como fator relevante: estudantes sem reprovacao tiveram media **12.51**, enquanto estudantes com reprovacao anterior tiveram media **8.59**. O tempo de estudo mostrou tendencia positiva, saindo de media **10.84** no menor nivel para **13.23** no nivel 3.

O apoio familiar apresentou diferenca moderada: a aprovacao foi **85.68%** com apoio e **82.87%** sem apoio. O consumo de alcool teve associacao negativa fraca com a nota final, com correlacao de **-0.209** para o indice combinado.

O indice de risco academico foi util para segmentar estudantes: o grupo de risco alto teve media **8.39** e aprovacao de **41.73%**, abaixo do grupo de risco baixo. Como o indice usa `G1` e `G2`, faltas e reprovacoes anteriores, ele evita usar diretamente a propria nota final como criterio de risco.

**Limitacoes:** os dados sao observacionais e nao permitem afirmar causalidade. Alem disso, a base representa estudantes portugueses de um contexto especifico, nao necessariamente todos os estudantes. Algumas variaveis importantes, como renda familiar, qualidade das aulas e historico individual detalhado, nao estao disponiveis.

**Proximos passos:** aplicar modelos de machine learning para prever risco de reprovacao, comparar Portugues e Matematica com cuidado para estudantes em comum e investigar intervencoes especificas para estudantes com reprovacao anterior e notas baixas nos primeiros periodos.
